In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
from pathlib import Path
import yaml
with open("/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/config/config.yaml") as f:
    _cfg = yaml.safe_load(f)
PROJ_DIR = Path(_cfg["ProjDIR"])
sys.path.insert(0, str(PROJ_DIR / "src"))
sys.path.insert(1, '/home/jw3514/Work/UNIMED/src')
from CellType_PSY import *
#from UNIMED import *

config = _cfg

#import scanpy as sc
HGNC, ENSID2Entrez, GeneSymbol2Entrez, Entrez2Symbol = LoadGeneINFO()

In [ ]:
# === Configuration flags ===
FORCE_RERUN = False       # Set True to recompute even if cache exists
SHOW_UNWEIGHTED = False   # Set True to show unweighted profiles

import pickle
_CACHE_DIR = PROJ_DIR / "results" / "figures" / "plot_data"
_CACHE_DIR.mkdir(parents=True, exist_ok=True)
_SWEEP_CACHE = _CACHE_DIR / "sweep_cache.pkl"
_EXP_CACHE = _CACHE_DIR / "expansion_slice_cache.pkl"

In [ ]:
# Load expression matrix
expression_matrix = config['analysis_types']['Centering']
print(expression_matrix)

In [ ]:
HumanCT_Z2_HCT = pd.read_csv(str(PROJ_DIR / expression_matrix), index_col=0)
HumanCT_Z2_HCT.columns = HumanCT_Z2_HCT.columns.astype(int)
HumanCT_Z2_HCT.shape

In [ ]:
GeneWeightDIR = str(PROJ_DIR / "dat" / "GeneWeights") + "/"
Bias_Save_Dir = str(PROJ_DIR / "results/main_results/random/Centering/")
if not os.path.exists(Bias_Save_Dir): # make dir if not exists
    os.makedirs(Bias_Save_Dir)

# Load BGMR for consistent gene weight correction across all disorders
BGMR = pd.read_csv("/home/jw3514/Work/Resources/BGMR.withEntrez.csv", low_memory=False)
BGMR["entrez_id"] = BGMR["entrez_id"].astype(int)
BGMR = BGMR.set_index("entrez_id")

In [ ]:
# Compute SCZ reference inline from top-61 of the sweep's gene weight file
# (ensures R=1 at N=61; pipeline file has only 53 genes after nopLI/exclude_Mis2 filtering)

In [ ]:
# The file has no header row, so treat all rows as data and provide a custom header
custom_header = ["Weight"]
GeneWeights = pd.read_csv(
    f"{GeneWeightDIR}/SCZ.top500.nopLI.LGD_Dmis_SameWeight.exclude_Mis2.gw",
    index_col=0,
    header=None,
    names=custom_header,
)

GeneWeights.head()

In [ ]:
GeneDF = pd.read_csv(str(PROJ_DIR / "dat/SCZ.ALLGENE.MutCountModified.csv"), index_col=0)

In [ ]:
for i, row in GeneWeights.iterrows():
    if i in GeneDF.index.values:
        GeneWeights.loc[i, "Pval"] = GeneDF.loc[i, "P meta"]
        GeneWeights.loc[i, "FDR"] = GeneDF.loc[i, "Q meta"]
        GeneWeights.loc[i, "Gene Symbol"] = GeneDF.loc[i, "Gene Symbol"]

In [ ]:
# Reference: top-61 from the same gene weight file used in the sweep
N_REF = 61
ref_SCZ_gw = dict(zip(GeneWeights.index[:N_REF], GeneWeights["Weight"].values[:N_REF]))
SCZ_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, ref_SCZ_gw)
SCZ_Bias = AnnotateCTDat(SCZ_Bias, Anno)

In [ ]:
# Calculate correlation as genes are removed, sorted by LofZ (descending)
GeneIdx = np.arange(10, 200)
if _SWEEP_CACHE.exists() and not FORCE_RERUN:
    import pickle as _pkl
    with open(_SWEEP_CACHE, "rb") as _f:
        _sw = _pkl.load(_f)
    Corr_SCZ, Pval_SCZ, Corr_SCZ_unweighted = _sw["SCZ"]
    print("SCZ sweep: from cache")
else:
    Corr_SCZ, Pval_SCZ, Corr_SCZ_unweighted = [], [], []
    for i in GeneIdx:
        tmp_SCZ_GW = dict(zip(GeneWeights.index.values[:i], GeneWeights["Weight"].values[:i]))
        tmp_SCZ_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, tmp_SCZ_GW)
        tmp_SCZ_Bias = AnnotateCTDat(tmp_SCZ_Bias, Anno)
        r, p = GetSingeCellBiasCorr(SCZ_Bias, tmp_SCZ_Bias, efflabel="EFFECT")
        
        Corr_SCZ.append(r)
        Pval_SCZ.append(p)
    
        # Unweighted version (each gene has weight 1)
        tmp_SCZ_GW_unweighted = dict(zip(GeneWeights.index.values[:i], np.ones(i)))
        tmp_SCZ_Bias_unweighted = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, tmp_SCZ_GW_unweighted)
        tmp_SCZ_Bias_unweighted = AnnotateCTDat(tmp_SCZ_Bias_unweighted, Anno)
        r_unweighted, _ = GetSingeCellBiasCorr(SCZ_Bias, tmp_SCZ_Bias_unweighted, efflabel="EFFECT")
        Corr_SCZ_unweighted.append(r_unweighted)

In [ ]:
import matplotlib.ticker as mticker

def plot_gene_set_correlation(
    GeneIdx_top,
    Corr_top,
    GeneSig_top,
    corr_label="Bias Correlation",
    pval_label=r"$-\log_{10}$(min p-value of Included Genes)",
    xlabel="Number of Genes",
    ylabel_corr="Bias Correlation with Main ASD with ID Set",
    ylabel_pval=None,
    figsize=(8, 4),
    dpi=120,
    color_corr='#1f77b4',
    color_pval='#d62728',
    legend_loc='center left',
    legend_bbox_to_anchor=(0.00, 0.1),
    legend_position=None,
    legend_fontsize=10,
    legend_kwargs=None,
    subplots_adjust_right=None,
    ylim_corr=(0, 1.01),
    nbins_x=8,
    title=None,
    ax1=None,
    ax2=None,
    show=True,
    Corr_unweighted=None,
    corr_unweighted_label="Bias Correlation (Unweighted)",
    color_unweighted='#2ca02c',
):
    """
    Plot gene set correlation and -log10(p-value) as a function of number of genes.

    Parameters
    ----------
    GeneIdx_top : array-like
        X-axis values (number of genes included).
    Corr_top : array-like
        Correlation values to plot on left y-axis.
    GeneSig_top : array-like
        -log10(p-value) values to plot on right y-axis.
    corr_label : str
        Label for the correlation line.
    pval_label : str
        Label for the p-value line.
    xlabel : str
        X-axis label.
    ylabel_corr : str
        Y-axis label for correlation.
    ylabel_pval : str or None
        Y-axis label for p-value. If None, uses pval_label.
    figsize : tuple
        Figure size.
    dpi : int
        Figure DPI.
    color_corr : str
        Color for correlation line.
    color_pval : str
        Color for p-value line.
    legend_loc : str
        Matplotlib ``loc`` for the legend (used when ``legend_position`` is None).
    legend_bbox_to_anchor : tuple or None
        Axes-fraction anchor (e.g. ``(1.02, 0.5)`` outside the right spine). None =
        legend inside the axes at ``legend_loc``.
    legend_position : str, tuple, or None
        If set, overrides ``legend_loc`` and ``legend_bbox_to_anchor``. ``None`` uses
        those two. A ``str`` is a matplotlib ``loc`` (inside axes). Presets:
        ``"outside_right"``, ``"outside_left"``. Or pass ``(loc, bbox_to_anchor)``;
        use ``None`` as bbox for in-axes only.
    legend_fontsize : int
        Font size for legend.
    legend_kwargs : dict or None
        Extra arguments passed to ``ax.legend`` (e.g. ``ncol``).
    subplots_adjust_right : float or None
        ``plt.subplots_adjust(right=...)``. If None, uses a tighter margin when the
        legend is outside the axes.
    ylim_corr : tuple
        Y-axis limits for correlation.
    nbins_x : int
        Number of bins for x-axis major ticks.
    title : str or None
        Optional plot title.
    ax1, ax2 : matplotlib axes or None
        Optionally provide axes to plot on.
    show : bool
        Whether to call plt.show().
    """
    import matplotlib.pyplot as plt

    _legend_presets = {
        "outside_right": ("center left", (1.02, 0.5)),
        "outside_left": ("center right", (-0.02, 0.5)),
    }

    if legend_position is not None:
        if isinstance(legend_position, str):
            if legend_position in _legend_presets:
                _leg_loc, _leg_bbox = _legend_presets[legend_position]
            else:
                _leg_loc, _leg_bbox = legend_position, None
        elif isinstance(legend_position, tuple) and len(legend_position) == 2:
            _leg_loc, _leg_bbox = legend_position
        else:
            raise TypeError(
                "legend_position must be None, a str, or a (loc, bbox_to_anchor) tuple"
            )
    else:
        _leg_loc, _leg_bbox = legend_loc, legend_bbox_to_anchor

    if ax1 is None or ax2 is None:
        fig, ax1 = plt.subplots(figsize=figsize, dpi=dpi)
    else:
        fig = ax1.figure

    # Plot correlation
    ax1.plot(GeneIdx_top, Corr_top, color=color_corr, linewidth=2, marker='o', markersize=4, label=corr_label)
    
    # Plot unweighted correlation if provided
    if Corr_unweighted is not None:
        ax1.plot(GeneIdx_top, Corr_unweighted, color=color_unweighted, linewidth=2, marker='^', markersize=4, label=corr_unweighted_label, linestyle='--')
    
    ax1.set_xlabel(xlabel, fontsize=12)
    ax1.set_ylabel(ylabel_corr, color=color_corr, fontsize=12)
    ax1.tick_params(axis='y', labelcolor=color_corr)
    ax1.set_ylim(*ylim_corr)
    ax1.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
    ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=nbins_x))
    ax1.xaxis.set_minor_locator(mticker.AutoMinorLocator())
    ax1.xaxis.set_minor_formatter(mticker.NullFormatter())

    # Only draw horizontal grid lines, and only behind the lines, not over text
    ax1.grid(True, which='major', axis='y', linestyle='--', alpha=0.5, zorder=1)
    # Remove x grid to avoid covering x-labels/ticks
    ax1.grid(False, which='major', axis='x', zorder=1, alpha=0.5)

    # Plot -log10(P-value) on secondary axis
    if ax2 is None:
        ax2 = ax1.twinx()
    ax2.plot(GeneIdx_top, GeneSig_top, color=color_pval, linewidth=2, marker='s', markersize=4, label=pval_label, zorder=3)
    ax2.set_ylabel(ylabel_pval if ylabel_pval is not None else pval_label, color=color_pval, fontsize=12)
    ax2.tick_params(axis='y', labelcolor=color_pval)
    ax2.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    _lk = dict(fontsize=legend_fontsize)
    if _leg_bbox is not None:
        _lk["bbox_to_anchor"] = _leg_bbox
        _lk.update(frameon=True, framealpha=0.96, edgecolor="0.85")
    else:
        _lk["frameon"] = False
    if legend_kwargs:
        _lk.update(legend_kwargs)
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc=_leg_loc, **_lk)

    # Publication style tweaks
    fig.tight_layout(pad=2)
    for spine in ax1.spines.values():
        spine.set_linewidth(1.2)
    for spine in ax2.spines.values():
        spine.set_linewidth(1.2)
    if subplots_adjust_right is not None:
        _adj_r = subplots_adjust_right
    else:
        _adj_r = 0.72 if _leg_bbox is not None else 0.88
    plt.subplots_adjust(top=0.95, right=_adj_r)
    if title is not None:
        ax1.set_title(title, fontsize=14, pad=10)
    if show:
        plt.show()
    return fig, ax1, ax2

In [ ]:
_fig_scz, _, _ = plot_gene_set_correlation(
    GeneIdx_top=GeneIdx,
    Corr_top=Corr_SCZ,
    GeneSig_top=-np.log10(np.array(GeneWeights["Pval"].values[GeneIdx], dtype=np.float64)),
    corr_label="Bias Correlation",
    ylabel_corr = "Bias Correlation with Main SCZ Set",
    pval_label=r"$-\log_{10}$(max p-value of Included Genes)",
    Corr_unweighted=Corr_SCZ_unweighted if SHOW_UNWEIGHTED else None,
    corr_unweighted_label="Bias Correlation (Unweighted)",
    show=False,
)
_figdir = str(PROJ_DIR / "results" / "figures") + "/"
os.makedirs(_figdir, exist_ok=True)
_fig_scz.savefig(_figdir + "gene_sweep_SCZ.png", dpi=300, bbox_inches='tight', transparent=True, facecolor='none')
plt.show()

# ASD

In [ ]:
# ASD references will be computed inline after weights are built (below)

In [ ]:
Spark_Denovo = pd.read_excel(str(PROJ_DIR / "dat/suppl.data/41588_2022_1148_MOESM4_ESM.xlsx"),
                           skiprows=2, sheet_name="Table S7")
Spark_Denovo = Spark_Denovo[Spark_Denovo[
    "pDenovoWEST_Meta"]!="."]
Spark_Denovo.shape

In [ ]:
Mut_n_IQ = pd.read_csv(str(PROJ_DIR / "dat/ASD_IQ_Mut.csv"))
HighIQMuts_ALL = Mut_n_IQ[Mut_n_IQ["IQ"]>70]
LowIQMuts_ALL = Mut_n_IQ[Mut_n_IQ["IQ"]<=70]

In [ ]:
Spark_Denovo_sub = Spark_Denovo[["EntrezID", "HGNC", "pDenovoWEST_Meta", "AutismMerged_LoF", "AutismMerged_Dmis_REVEL0.5"]]
Spark_Denovo_sub = Spark_Denovo_sub.set_index("EntrezID")

# Count IQ-specific mutations per gene: total + LoF/Dmis split for BGMR correction
lgd_types = ["frameshift", "splice_acceptor", "splice_donor", "start_lost", "stop_gained", "stop_lost"]
for i in Spark_Denovo_sub.index:
    hiq_muts = HighIQMuts_ALL[HighIQMuts_ALL["Entrez"] == i]
    liq_muts = LowIQMuts_ALL[LowIQMuts_ALL["Entrez"] == i]
    Spark_Denovo_sub.loc[i, "HIQ_counts"] = len(hiq_muts)
    Spark_Denovo_sub.loc[i, "LIQ_counts"] = len(liq_muts)
    for prefix, muts in [("HIQ", hiq_muts), ("LIQ", liq_muts)]:
        eff = muts["GeneEff"].str.split(";").str[0]
        n_lgd = eff.isin(lgd_types).sum()
        is_mis = eff == "missense"
        revel = muts.loc[is_mis, "REVEL"].str.split(";").str[0]
        n_dmis = revel.apply(lambda x: float(x) > 0.5 if x != "." else False).sum()
        Spark_Denovo_sub.loc[i, f"{prefix}_LGD"] = n_lgd
        Spark_Denovo_sub.loc[i, f"{prefix}_Dmis"] = n_dmis
    if Spark_Denovo_sub.loc[i, "pDenovoWEST_Meta"] == 0:
        Spark_Denovo_sub.loc[i, "pDenovoWEST_Meta"] = 1e-6

print(f"Spark_Denovo_sub: {Spark_Denovo_sub.shape}")
print(f"HIQ genes with mutations: {(Spark_Denovo_sub['HIQ_counts'] > 0).sum()}")
print(f"LIQ genes with mutations: {(Spark_Denovo_sub['LIQ_counts'] > 0).sum()}")

In [ ]:
# Compute BGMR-corrected ASD gene weights for ALL ranked genes (not just top-61).
from collections import OrderedDict

N_ASD_HIQ = 3610
N_ASD_LIQ = 1792


def _asd_iq_bgmr_weights(spark_sub, iq_prefix, N_proband, n_top):
    """Compute BGMR-corrected weights for ASD IQ subgroup."""
    top = spark_sub.head(n_top)
    top = top[top[f"{iq_prefix}_counts"] > 0]
    gw = OrderedDict()
    for g in top.index:
        if g not in BGMR.index:
            continue
        nLGD = float(top.loc[g, f"{iq_prefix}_LGD"])
        nDmis = float(top.loc[g, f"{iq_prefix}_Dmis"])
        exp_LGD = BGMR.loc[g, "p_LGD"] * 2 * N_proband
        exp_Dmis = BGMR.loc[g, "prevel_0.5"] * 2 * N_proband
        weight = (nLGD - exp_LGD) + (nDmis - exp_Dmis)
        gw[g] = max(weight, 0)
    return gw


N_SWEEP = len(Spark_Denovo_sub)
hiq_all_gw = _asd_iq_bgmr_weights(Spark_Denovo_sub, "HIQ", N_ASD_HIQ, N_SWEEP)
liq_all_gw = _asd_iq_bgmr_weights(Spark_Denovo_sub, "LIQ", N_ASD_LIQ, N_SWEEP)

Spark_Denovo_sub["HIQ_weight_bgmr"] = 0.0
Spark_Denovo_sub["LIQ_weight_bgmr"] = 0.0
for g, w in hiq_all_gw.items():
    Spark_Denovo_sub.loc[g, "HIQ_weight_bgmr"] = w
for g, w in liq_all_gw.items():
    Spark_Denovo_sub.loc[g, "LIQ_weight_bgmr"] = w

print(f"HIQ BGMR weights: {len(hiq_all_gw)} genes with nonzero weight")
print(f"LIQ BGMR weights: {len(liq_all_gw)} genes with nonzero weight")

In [ ]:
# Compute ASD references inline from top-61 of the same gene weights used in the sweep
ref_HIQ_gw = dict(zip(Spark_Denovo_sub.index[:N_REF], Spark_Denovo_sub["HIQ_weight_bgmr"].values[:N_REF]))
HIQ_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, ref_HIQ_gw)
HIQ_Bias = AnnotateCTDat(HIQ_Bias, Anno)

ref_LIQ_gw = dict(zip(Spark_Denovo_sub.index[:N_REF], Spark_Denovo_sub["LIQ_weight_bgmr"].values[:N_REF]))
LIQ_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, ref_LIQ_gw)
LIQ_Bias = AnnotateCTDat(LIQ_Bias, Anno)

In [ ]:
# Calculate correlation as genes are added — HIQ (BGMR-corrected)
GeneIdx = np.arange(10, 200)
if _SWEEP_CACHE.exists() and not FORCE_RERUN:
    import pickle as _pkl
    with open(_SWEEP_CACHE, "rb") as _f:
        _sw = _pkl.load(_f)
    Corr_HIQ, Pval_HIQ, Corr_HIQ_unweighted = _sw["HIQ"]
    print("HIQ sweep: from cache")
else:
    Corr_HIQ, Pval_HIQ, Corr_HIQ_unweighted = [], [], []
    for i in GeneIdx:
        tmp_HIQ_ASD_GW = dict(zip(Spark_Denovo_sub.index.values[:i], Spark_Denovo_sub["HIQ_weight_bgmr"].values[:i]))
        tmp_HIQ_ASD_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, tmp_HIQ_ASD_GW)
        tmp_HIQ_ASD_Bias = AnnotateCTDat(tmp_HIQ_ASD_Bias, Anno)
        r, p = GetSingeCellBiasCorr(HIQ_Bias, tmp_HIQ_ASD_Bias, efflabel="EFFECT")

        Corr_HIQ.append(r)
        Pval_HIQ.append(p)

        # Unweighted version (each gene has weight 1)
        tmp_HIQ_ASD_GW_unweighted = dict(zip(Spark_Denovo_sub.index.values[:i], np.ones(i)))
        tmp_HIQ_ASD_Bias_unweighted = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, tmp_HIQ_ASD_GW_unweighted)
        tmp_HIQ_ASD_Bias_unweighted = AnnotateCTDat(tmp_HIQ_ASD_Bias_unweighted, Anno)
        r_unweighted, _ = GetSingeCellBiasCorr(HIQ_Bias, tmp_HIQ_ASD_Bias_unweighted, efflabel="EFFECT")
        Corr_HIQ_unweighted.append(r_unweighted)

In [ ]:
# Calculate correlation as genes are added — LIQ (BGMR-corrected)
GeneIdx = np.arange(10, 200)
if _SWEEP_CACHE.exists() and not FORCE_RERUN:
    import pickle as _pkl
    with open(_SWEEP_CACHE, "rb") as _f:
        _sw = _pkl.load(_f)
    Corr_LIQ, Pval_LIQ, Corr_LIQ_unweighted = _sw["LIQ"]
    print("LIQ sweep: from cache")
else:
    Corr_LIQ, Pval_LIQ, Corr_LIQ_unweighted = [], [], []
    for i in GeneIdx:
        tmp_LIQ_ASD_GW = dict(zip(Spark_Denovo_sub.index.values[:i], Spark_Denovo_sub["LIQ_weight_bgmr"].values[:i]))
        tmp_LIQ_ASD_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, tmp_LIQ_ASD_GW)
        tmp_LIQ_ASD_Bias = AnnotateCTDat(tmp_LIQ_ASD_Bias, Anno)
        r, p = GetSingeCellBiasCorr(LIQ_Bias, tmp_LIQ_ASD_Bias, efflabel="EFFECT")

        Corr_LIQ.append(r)
        Pval_LIQ.append(p)

        # Unweighted version (each gene has weight 1)
        tmp_LIQ_ASD_GW_unweighted = dict(zip(Spark_Denovo_sub.index.values[:i], np.ones(i)))
        tmp_LIQ_ASD_Bias_unweighted = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, tmp_LIQ_ASD_GW_unweighted)
        tmp_LIQ_ASD_Bias_unweighted = AnnotateCTDat(tmp_LIQ_ASD_Bias_unweighted, Anno)
        r_unweighted, _ = GetSingeCellBiasCorr(LIQ_Bias, tmp_LIQ_ASD_Bias_unweighted, efflabel="EFFECT")
        Corr_LIQ_unweighted.append(r_unweighted)

In [ ]:
_fig_hiq, _, _ = plot_gene_set_correlation(
    GeneIdx_top=GeneIdx,
    Corr_top=Corr_HIQ,
    GeneSig_top=-np.log10(np.array(Spark_Denovo_sub["pDenovoWEST_Meta"].values[GeneIdx], dtype=np.float64)),
    corr_label="Bias Correlation",
    ylabel_corr = "Bias Correlation with \nMain ASD w/o IDSet",
    pval_label=r"$-\log_{10}$(max p-value of Included Genes)",
    Corr_unweighted=Corr_HIQ_unweighted if SHOW_UNWEIGHTED else None,
    corr_unweighted_label="Bias Correlation (Unweighted)",
    show=False,
)
_fig_hiq.savefig(_figdir + "gene_sweep_ASD_woID.png", dpi=300, bbox_inches='tight', transparent=True, facecolor='none')
plt.show()

_fig_liq, _, _ = plot_gene_set_correlation(
    GeneIdx_top=GeneIdx,
    Corr_top=Corr_LIQ,
    GeneSig_top=-np.log10(np.array(Spark_Denovo_sub["pDenovoWEST_Meta"].values[GeneIdx], dtype=np.float64)),
    corr_label="Bias Correlation",
    ylabel_corr = "Bias Correlation with Main \n ASD with ID Set",
    pval_label=r"$-\log_{10}$(max p-value of Included Genes)",
    Corr_unweighted=Corr_LIQ_unweighted if SHOW_UNWEIGHTED else None,
    corr_unweighted_label="Bias Correlation (Unweighted)",
    show=False,
)
_fig_liq.savefig(_figdir + "gene_sweep_ASD_wID.png", dpi=300, bbox_inches='tight', transparent=True, facecolor='none')
plt.show()

# DDD

In [ ]:
# DDD reference will be computed inline after DDD_Genes_sub is built (below)

In [ ]:
DDD_Genes = pd.read_excel("/home/jw3514/Work/data/DDD/41586_2020_2832_MOESM4_ESM.xlsx")
DDD_Genes = DDD_Genes.sort_values("denovoWEST_p_full")
entrez_ids = [int(GeneSymbol2Entrez.get(x, -1)) for x in DDD_Genes["symbol"].values]
DDD_Genes["EntrezID"] = entrez_ids
DDD_Genes.shape

In [ ]:
DDD_Genes_sub = DDD_Genes[["symbol", "EntrezID", "denovoWEST_p_full", "missense_variant", "frameshift_variant", "splice_acceptor_variant", "splice_donor_variant", "stop_gained", "stop_lost"]].reset_index(drop=True)
# Keep all genes — don't filter by BGMR membership. Genes without valid EntrezID
# or BGMR data get weight=0 (padded), preserving the true significance rank count.

In [ ]:
# Helper: compute BGMR-corrected weights, padding invalid genes with weight=0
def _ddd_gene_weights(df, Nproband, BGMR):
    """Compute DDD gene weights; genes not in BGMR get weight=0 (preserves rank count)."""
    valid = df[df["EntrezID"].isin(BGMR.index)]
    if len(valid) > 0:
        return Aggregate_Gene_Weights_NDD(valid, Nproband=Nproband, BGMR=BGMR, wLGD=1, wMis=1)
    return {}

In [ ]:
# Compute DDD reference inline from top-61 (preserving original rank, padding filtered genes)
N_DDD = 31058
ref_DDD_gw = _ddd_gene_weights(DDD_Genes_sub.head(N_REF), Nproband=N_DDD, BGMR=BGMR)
DDD_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, ref_DDD_gw)
DDD_Bias = AnnotateCTDat(DDD_Bias, Anno)

In [ ]:
# Calculate correlation as genes are added, sorted by significance (descending)
GeneIdx = np.arange(10, 200)
if _SWEEP_CACHE.exists() and not FORCE_RERUN:
    import pickle as _pkl
    with open(_SWEEP_CACHE, "rb") as _f:
        _sw = _pkl.load(_f)
    Corr_DDD, Pval_DDD, Corr_DDD_unweighted = _sw["DDD"]
    print("DDD sweep: from cache")
else:
    Corr_DDD, Pval_DDD, Corr_DDD_unweighted = [], [], []
    for i in GeneIdx:
        tmp = DDD_Genes_sub.head(i)
        tmp_DDD_GW = _ddd_gene_weights(tmp, Nproband=N_DDD, BGMR=BGMR)
        tmp_DDD_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, tmp_DDD_GW)
        tmp_DDD_Bias = AnnotateCTDat(tmp_DDD_Bias, Anno)
        r, p = GetSingeCellBiasCorr(DDD_Bias, tmp_DDD_Bias, efflabel="EFFECT")

        Corr_DDD.append(r)
        Pval_DDD.append(p)

        # Unweighted version: valid genes get weight=1, invalid get weight=0
        valid_ids = tmp[tmp["EntrezID"].isin(BGMR.index)]["EntrezID"].values
        tmp_DDD_GW_unweighted = dict(zip(valid_ids, np.ones(len(valid_ids))))
        tmp_DDD_Bias_unweighted = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, tmp_DDD_GW_unweighted)
        tmp_DDD_Bias_unweighted = AnnotateCTDat(tmp_DDD_Bias_unweighted, Anno)
        r_unweighted, _ = GetSingeCellBiasCorr(DDD_Bias, tmp_DDD_Bias_unweighted, efflabel="EFFECT")
        Corr_DDD_unweighted.append(r_unweighted)

In [ ]:
_fig_ddd, _, _ = plot_gene_set_correlation(
    GeneIdx_top=GeneIdx,
    Corr_top=Corr_DDD,
    GeneSig_top=-np.log10(np.array(DDD_Genes["denovoWEST_p_full"].values[GeneIdx], dtype=np.float64)),
    corr_label="Bias Correlation",
    ylabel_corr = "Bias Correlation with Main DDD Set",
    pval_label=r"$-\log_{10}$(max p-value of Included Genes)",
    Corr_unweighted=Corr_DDD_unweighted if SHOW_UNWEIGHTED else None,
    corr_unweighted_label="Bias Correlation (Unweighted)",
    show=False,
)
_fig_ddd.savefig(_figdir + "gene_sweep_DDD.png", dpi=300, bbox_inches='tight', transparent=True, facecolor='none')
plt.show()

# Reviewer Figure R3.2a: SCZ Split-Half Bias Comparison

Do added SCZ genes (62-200) carry concordant cell-type signal,
or are they noise? Compare bias profiles of top-61 vs added vs random.

Supports reply to Reviewer #3, Point 2, Paragraph 1 (concordant signal).

In [ ]:
from scipy.stats import spearmanr

# Compute bias for three gene subsets
top61_gw = dict(zip(GeneWeights.index[:61], GeneWeights["Weight"].values[:61]))
added_gw = dict(zip(GeneWeights.index[61:200], GeneWeights["Weight"].values[61:200]))

top61_bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, top61_gw)
top61_bias = AnnotateCTDat(top61_bias, Anno)

added_bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, added_gw)
added_bias = AnnotateCTDat(added_bias, Anno)

# Random control: 139 genes not in SCZ top-200, with average weight of added genes
np.random.seed(42)
exclude_set = set(GeneWeights.index[:200].astype(int))
gene_pool = [g for g in HumanCT_Z2_HCT.index.values if g not in exclude_set]
random_genes = np.random.choice(gene_pool, size=139, replace=False)
avg_added_weight = GeneWeights["Weight"].values[61:200].mean()
random_gw = dict(zip(random_genes, np.ones(139) * avg_added_weight))
random_bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, random_gw)
random_bias = AnnotateCTDat(random_bias, Anno)

# Correlations
common_idx = top61_bias.index
neur_mask = top61_bias.index.isin(Neur_idx)

r_added_all, p_added_all = spearmanr(top61_bias["EFFECT"], added_bias.loc[common_idx, "EFFECT"])
r_added_neur, p_added_neur = spearmanr(top61_bias.loc[neur_mask, "EFFECT"], added_bias.loc[top61_bias.loc[neur_mask].index, "EFFECT"])
r_rand_all, p_rand_all = spearmanr(top61_bias["EFFECT"], random_bias.loc[common_idx, "EFFECT"])

print(f"Top-61 vs Added (62-200):  all CTs r={r_added_all:.3f} (p={p_added_all:.2e}), neurons r={r_added_neur:.3f} (p={p_added_neur:.2e})")
print(f"Top-61 vs Random (n=139):  all CTs r={r_rand_all:.3f} (p={p_rand_all:.2e})")

print("\nTop 5 superclusters — Top-61 genes:")
for sc, val in top61_bias.groupby("Supercluster")["EFFECT"].mean().sort_values(ascending=False).head(5).items():
    print(f"  {sc}: {val:.4f}")

print("\nTop 5 superclusters — Added genes (62-200):")
for sc, val in added_bias.groupby("Supercluster")["EFFECT"].mean().sort_values(ascending=False).head(5).items():
    print(f"  {sc}: {val:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=120)

for ax_idx, (comp_bias, comp_label, r_val, p_val) in enumerate([
    (added_bias, "Added genes (62-200)", r_added_all, p_added_all),
    (random_bias, "Random genes (n=139)", r_rand_all, p_rand_all),
]):
    ax = axes[ax_idx]
    x = top61_bias["EFFECT"].values
    y = comp_bias.loc[common_idx, "EFFECT"].values

    ax.scatter(x[neur_mask], y[neur_mask], color="red", alpha=0.4, s=20,
               edgecolors="white", lw=0.3, label="Neuronal", zorder=3)
    ax.scatter(x[~neur_mask], y[~neur_mask], color="blue", alpha=0.5, s=20,
               edgecolors="white", lw=0.3, label="Non-neuronal", zorder=4)

    # Reference line
    lims = [min(x.min(), y.min()), max(x.max(), y.max())]
    ax.plot(lims, lims, "k--", lw=0.8, alpha=0.4)
    ax.axhline(0, color="gray", lw=0.5, alpha=0.3)
    ax.axvline(0, color="gray", lw=0.5, alpha=0.3)

    ax.set_xlabel("Top-61 SCZ genes — EFFECT", fontsize=11)
    ax.set_ylabel(f"{comp_label} — EFFECT", fontsize=11)
    ax.set_title(chr(65 + ax_idx), fontweight="bold", loc="left", fontsize=14)
    ax.legend(fontsize=9, framealpha=0.8)
    ax.text(0.97, 0.03, f"ρ = {r_val:.3f}\np = {p_val:.1e}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

fig.suptitle("R3.2a — SCZ: Added genes carry concordant cell-type signal", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.patch.set_alpha(0)
plt.show()

# Reviewer Figure R3.2b: Sliding Window Correlation Decay (SCZ)

Use a sliding window of 61 genes to show how far down the ranked list
the cell-type bias signal persists.

Supports reply to Reviewer #3, Point 2, Paragraph 1 (signal persistence).

In [ ]:
GeneDF = pd.read_csv(str(PROJ_DIR / "dat/SCZ.ALLGENE.MutCountModified.csv"), index_col=0)

# Add P-values to full gene weight list (if not already present)
for i in GeneWeights.index:
    if i in GeneDF.index.values:
        GeneWeights.loc[i, "Pval"] = GeneDF.loc[i, "P meta"]

window_size = 61
start_ranks = np.arange(0, 200 - window_size + 1, 10)
window_corrs = []
window_mean_pvals = []

for start in start_ranks:
    end = start + window_size
    w_gw = dict(zip(GeneWeights.index[start:end], GeneWeights["Weight"].values[start:end]))
    w_bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, w_gw)
    w_bias = AnnotateCTDat(w_bias, Anno)
    r, _ = spearmanr(top61_bias["EFFECT"], w_bias.loc[common_idx, "EFFECT"])
    window_corrs.append(r)

    pvals_in_window = GeneWeights.iloc[start:end]["Pval"].dropna().astype(float)
    window_mean_pvals.append(pvals_in_window.mean() if len(pvals_in_window) > 0 else np.nan)

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4.5), dpi=120)

color_corr = "#1f77b4"
color_pval = "#d62728"

ax1.plot(start_ranks + 1, window_corrs, color=color_corr, marker="o", markersize=5, lw=2, label="Spearmans' R with top-61")
ax1.set_xlabel("Starting gene rank", fontsize=12)
ax1.set_ylabel("Spearmans' R (window bias vs top-61 bias)", color=color_corr, fontsize=11)
ax1.tick_params(axis="y", labelcolor=color_corr)
ax1.set_ylim(-0.1, 1.05)
ax1.axhline(0, color="gray", lw=0.5, alpha=0.5)

ax2 = ax1.twinx()
ax2.plot(start_ranks + 1, -np.log10(np.array(window_mean_pvals)),
         color=color_pval, marker="s", markersize=4, lw=1.5, ls="--",
         label=r"$-\log_{10}$(mean P-value)")
ax2.set_ylabel(r"$-\log_{10}$(mean P-value in window)", color=color_pval, fontsize=11)
ax2.tick_params(axis="y", labelcolor=color_pval)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9, framealpha=0.8)

ax1.set_title("R3.2b — SCZ: Sliding window (61 genes) correlation decay", fontweight="bold", fontsize=12)
for spine in ["top"]:
    ax1.spines[spine].set_visible(False)
    ax2.spines[spine].set_visible(False)

fig.tight_layout()
fig.patch.set_alpha(0)
plt.show()

idx_below = np.searchsorted(-np.array(window_corrs), -0.3)
if idx_below < len(start_ranks):
    print(f"Signal persistence: ρ > 0.3 until rank ~{start_ranks[idx_below] + 1}")
else:
    print(f"Signal persistence: ρ > 0.3 across ALL windows (min ρ = {min(window_corrs):.3f})")

# Reviewer Figure R3.2c: Real vs Random Gene Additions (SCZ, ASD, DDD)

Does expanding each disorder's gene set beyond top-61 add signal or noise?
For each disorder we compare:
- **Real**: append the next ranked disease genes (62–N)
- **Random**: append random genes to the fixed top-61 core (100 reps, 95% CI)

Metric: Spearmans' R between expanded bias profile and top-61 reference.

In [ ]:
from joblib import Parallel, delayed

# Build gene weight DataFrames for R3.2c using the SAME gene ranking as sweeps (Panels A-D).
# Panel C/D sweeps use Spark_Denovo_sub (ASD) and DDD_Genes_sub (DDD) ranked by significance,
# with BGMR-corrected weights for top-61 and raw counts beyond.
# We build identical DataFrames here for consistency.

# ASD (all): BGMR-corrected weights for ALL ranked genes using full cohort
N_ASD_ALL = 42607
ASD_GW_full = pd.DataFrame({"Weight": 0.0}, index=Spark_Denovo_sub.index)
for g in Spark_Denovo_sub.index:
    if g not in BGMR.index:
        continue
    nLGD = float(Spark_Denovo_sub.loc[g, "AutismMerged_LoF"])
    nDmis = float(Spark_Denovo_sub.loc[g, "AutismMerged_Dmis_REVEL0.5"])
    exp_LGD = BGMR.loc[g, "p_LGD"] * 2 * N_ASD_ALL
    exp_Dmis = BGMR.loc[g, "prevel_0.5"] * 2 * N_ASD_ALL
    ASD_GW_full.loc[g, "Weight"] = max((nLGD - exp_LGD) + (nDmis - exp_Dmis), 0)

# DDD: BGMR-corrected weights for ALL genes
_ddd_all_gw = _ddd_gene_weights(DDD_Genes_sub, Nproband=N_DDD, BGMR=BGMR)
DDD_GW_full = pd.DataFrame([
    {"EntrezID": row["EntrezID"], "Weight": _ddd_all_gw.get(row["EntrezID"], 0)}
    for _, row in DDD_Genes_sub.iterrows()
    if row["EntrezID"] > 0
]).set_index("EntrezID")
DDD_GW_full["Weight"] = DDD_GW_full["Weight"].clip(lower=0)

n_asd_nz = (ASD_GW_full["Weight"] > 0).sum()
print(f"SCZ: {len(GeneWeights)} genes, ASD: {n_asd_nz} nonzero, DDD: {len(DDD_GW_full)} genes")

In [ ]:
n_add_values = np.array([0, 1, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140])
n_reps = 100

disorder_configs = [
    ("SCZ",  GeneWeights,  "#ff7f0e"),
    ("ASD",  ASD_GW_full,  "#1f77b4"),
    ("DDD",  DDD_GW_full,  "#2ca02c"),
]

if _EXP_CACHE.exists() and not FORCE_RERUN:
    with open(_EXP_CACHE, "rb") as _f:
        _exp = pickle.load(_f)
    all_results = _exp["all_results"]
    print("R3.2c expansion: from cache")
else:
    all_results = {}

    for disorder_name, gw_df, color in disorder_configs:
        print(f"\n{'='*50}")
        print(f"Processing {disorder_name} ({len(gw_df)} genes available)")

        # Reference: top-61 bias profile for this disorder
        n_ref = min(61, len(gw_df))
        ref_gw = dict(zip(gw_df.index[:n_ref], gw_df["Weight"].values[:n_ref]))
        ref_bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, ref_gw)
        ref_bias = AnnotateCTDat(ref_bias, Anno)
        ref_effect = ref_bias["EFFECT"].values
        common_idx = ref_bias.index

        # Gene pool for random: exclude top-200 of this disorder
        n_exclude = min(200, len(gw_df))
        exclude_set = set(gw_df.index[:n_exclude].astype(int))
        gene_pool = np.array([g for g in HumanCT_Z2_HCT.index.values if g not in exclude_set])

        # Full weight vector — only use genes with nonzero weight for expansion.
        # Zero-weight genes (no mutations) contribute nothing to bias but inflate n_add,
        # causing random noise (which averages to ~0 on mean-centered data) to appear
        # to "preserve" correlation better than real signal.
        all_weights = gw_df["Weight"].values

        # Build list of (index, weight) for genes beyond top-61 with nonzero weight
        added_pool = [(gw_df.index[i], all_weights[i])
                      for i in range(n_ref, len(gw_df)) if all_weights[i] > 0]
        added_indices = [x[0] for x in added_pool]
        added_weights = np.array([x[1] for x in added_pool])
        print(f"  Genes beyond top-{n_ref} with nonzero weight: {len(added_pool)}")

        # ---- Real ranked genes (only nonzero-weight) ----
        real_corrs = []
        for n_add in n_add_values:
            n_use = min(n_add, len(added_pool))
            gw = dict(zip(gw_df.index[:n_ref], all_weights[:n_ref]))
            gw.update(dict(zip(added_indices[:n_use], added_weights[:n_use])))
            bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, gw)
            bias = AnnotateCTDat(bias, Anno)
            r, _ = spearmanr(ref_effect, bias.loc[common_idx, "EFFECT"].values)
            real_corrs.append(r)

        # ---- Random gene additions (matched nonzero weights) ----
        # Random genes get the same weights as the real nonzero-weight genes,
        # so the only difference is gene identity.
        def _one_random_trial(n_use, seed, top61_idx, top61_wt, gene_pool, rank_weights):
            rng = np.random.default_rng(seed)
            rand_genes = rng.choice(gene_pool, size=n_use, replace=False)
            gw = dict(zip(top61_idx, top61_wt))
            gw.update(dict(zip(rand_genes, rank_weights[:n_use])))
            bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, gw)
            bias = AnnotateCTDat(bias, Anno)
            r, _ = spearmanr(ref_effect, bias.loc[common_idx, "EFFECT"].values)
            return r

        top61_idx = gw_df.index[:n_ref]
        top61_wt = gw_df["Weight"].values[:n_ref]

        rand_mean = [1.0]
        rand_lo = [1.0]
        rand_hi = [1.0]

        for n_add in n_add_values[1:]:
            n_use = min(n_add, len(added_pool))
            if n_use == 0:
                rand_mean.append(1.0)
                rand_lo.append(1.0)
                rand_hi.append(1.0)
                continue
            rs = Parallel(n_jobs=10)(
                delayed(_one_random_trial)(
                    n_use, seed=hash((disorder_name, n_add, rep)) % (2**31),
                    top61_idx=top61_idx, top61_wt=top61_wt,
                    gene_pool=gene_pool, rank_weights=added_weights,
                )
                for rep in range(n_reps)
            )
            rs = np.array(rs)
            rand_mean.append(rs.mean())
            rand_lo.append(np.percentile(rs, 2.5))
            rand_hi.append(np.percentile(rs, 97.5))

        all_results[disorder_name] = {
            "real": np.array(real_corrs),
            "rand_mean": np.array(rand_mean),
            "rand_lo": np.array(rand_lo),
            "rand_hi": np.array(rand_hi),
            "color": color,
        }
        print(f"  Done. Real ρ at N=201: {real_corrs[-1]:.3f}, Random mean: {rand_mean[-1]:.3f}")

In [ ]:
total_genes = 61 + n_add_values

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=120, sharey=True)

for ax, (disorder_name, gw_df, color) in zip(axes, disorder_configs):
    res = all_results[disorder_name]

    # Random: mean + 95% CI band
    ax.fill_between(total_genes, res["rand_lo"], res["rand_hi"],
                    color="#999999", alpha=0.25, label="Random genes (95% CI)")
    ax.plot(total_genes, res["rand_mean"], color="#999999", lw=2, ls="--",
            marker="s", markersize=3, label="Random genes (mean)")

    # Real ranked genes
    ax.plot(total_genes, res["real"], color=color, lw=2.5,
            marker="o", markersize=5, label=f"Ranked {disorder_name} genes", zorder=5)

    ax.axvline(61, color="gray", ls=":", lw=1.5, alpha=0.5)
    ax.set_xlabel("Total number of genes", fontsize=12)
    ax.set_title(disorder_name, fontweight="bold", fontsize=14)
    ax.legend(fontsize=8, framealpha=0.8, loc="lower left")
    ax.set_ylim(0.3, 1.02)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

axes[0].set_ylabel("Spearmans' R with top-61 bias profile", fontsize=12)

fig.suptitle("R3.2c — Expanding gene sets: real ranked genes vs random additions",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.patch.set_alpha(0)
_figdir = str(PROJ_DIR / "results" / "figures") + "/"
os.makedirs(_figdir, exist_ok=True)
fig.savefig(_figdir + "FigSX_gene_expansion_R3.2c.pdf",
            dpi=300, bbox_inches='tight', transparent=True, facecolor='none')
fig.savefig(_figdir + "FigSX_gene_expansion_R3.2c.png",
            dpi=300, bbox_inches='tight', transparent=True, facecolor='none')
print(f"Saved: {_figdir}FigSX_gene_expansion_R3.2c.pdf/png")
plt.show()

## R3.2c-slice: Incremental Gene Correlation (Slice Only, No Top-61 Core)

The combined-set figure above is confounded by a dilution-vs-distortion effect
on mean-centered data: random genes contribute ~zero signal, so the combined
bias is just the top-61 scaled down (preserving rank → high correlation).

This panel computes bias from ONLY the incremental slice (no top-61 core)
and correlates with the top-61 reference. This directly tests whether the
added genes carry concordant cell-type signal.

In [ ]:
# Compute slice-only correlations for all disorders
if _EXP_CACHE.exists() and not FORCE_RERUN and "slice_results" in pickle.load(open(_EXP_CACHE,"rb")):
    with open(_EXP_CACHE, "rb") as _f:
        slice_results = pickle.load(_f)["slice_results"]
    print("Slice analysis: from cache")
else:
    slice_results = {}

    for disorder_name, gw_df, color in disorder_configs:
        print(f"\n{'='*50}")
        print(f"Slice analysis: {disorder_name}")

        n_ref = min(61, len(gw_df))
        ref_gw = dict(zip(gw_df.index[:n_ref], gw_df["Weight"].values[:n_ref]))
        ref_bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, ref_gw)
        ref_bias = AnnotateCTDat(ref_bias, Anno)
        ref_effect = ref_bias["EFFECT"].values
        common_idx = ref_bias.index

        all_weights = gw_df["Weight"].values
        added_pool = [(gw_df.index[i], all_weights[i])
                      for i in range(n_ref, len(gw_df)) if all_weights[i] > 0]
        added_indices = [x[0] for x in added_pool]
        added_weights = np.array([x[1] for x in added_pool])

        n_exclude = min(200, len(gw_df))
        exclude_set = set(gw_df.index[:n_exclude].astype(int))
        gene_pool = np.array([g for g in HumanCT_Z2_HCT.index.values if g not in exclude_set])

        # ---- Real slice only (no top-61 core) ----
        max_real_add = len(added_pool)
        real_slice_corrs = [np.nan]  # n_add=0 → no genes → undefined
        for n_add in n_add_values[1:]:
            if n_add > max_real_add:
                real_slice_corrs.append(np.nan)  # pool exhausted
                continue
            n_use = min(n_add, max_real_add)
            if n_use == 0:
                real_slice_corrs.append(np.nan)
                continue
            gw = dict(zip(added_indices[:n_use], added_weights[:n_use]))
            bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, gw)
            bias = AnnotateCTDat(bias, Anno)
            r, _ = spearmanr(ref_effect, bias.loc[common_idx, "EFFECT"].values)
            real_slice_corrs.append(r)
        print(f"  {disorder_name}: {max_real_add} nonzero-weight genes beyond top-61")

        # ---- Random slice only (no top-61 core) ----
        def _one_random_slice_trial(n_use, seed, gene_pool, rank_weights, ref_effect, common_idx):
            rng = np.random.default_rng(seed)
            rand_genes = rng.choice(gene_pool, size=n_use, replace=False)
            gw = dict(zip(rand_genes, rank_weights[:n_use]))
            bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, gw)
            bias = AnnotateCTDat(bias, Anno)
            r, _ = spearmanr(ref_effect, bias.loc[common_idx, "EFFECT"].values)
            return r

        rand_slice_mean = [np.nan]
        rand_slice_lo = [np.nan]
        rand_slice_hi = [np.nan]

        for n_add in n_add_values[1:]:
            n_use = min(n_add, len(added_pool))
            if n_use == 0:
                rand_slice_mean.append(np.nan)
                rand_slice_lo.append(np.nan)
                rand_slice_hi.append(np.nan)
                continue
            rs = Parallel(n_jobs=10)(
                delayed(_one_random_slice_trial)(
                    n_use, seed=hash((disorder_name, "slice", n_add, rep)) % (2**31),
                    gene_pool=gene_pool, rank_weights=added_weights,
                    ref_effect=ref_effect, common_idx=common_idx,
                )
                for rep in range(n_reps)
            )
            rs = np.array(rs)
            rand_slice_mean.append(rs.mean())
            rand_slice_lo.append(np.percentile(rs, 2.5))
            rand_slice_hi.append(np.percentile(rs, 97.5))

        slice_results[disorder_name] = {
            "real": np.array(real_slice_corrs, dtype=float),
            "rand_mean": np.array(rand_slice_mean, dtype=float),
            "rand_lo": np.array(rand_slice_lo, dtype=float),
            "rand_hi": np.array(rand_slice_hi, dtype=float),
            "color": color,
        }
        print(f"  Real slice at N=140: rho={real_slice_corrs[-1]:.3f}, "
              f"Random mean: {rand_slice_mean[-1]:.3f}")

In [ ]:
# Plot slice-only panels with fully transparent background
fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=120, sharey=True)

for ax, (disorder_name, gw_df, color) in zip(axes, disorder_configs):
    res = slice_results[disorder_name]
    valid = ~np.isnan(res["real"])
    x = n_add_values[valid]

    # Random: mean + 95% CI band
    ax.fill_between(x, res["rand_lo"][valid], res["rand_hi"][valid],
                    color="#999999", alpha=0.25, label="Random genes (95% CI)")
    ax.plot(x, res["rand_mean"][valid], color="#999999", lw=2, ls="--",
            marker="s", markersize=4.5, label="Random genes (mean)")  # 3 * 1.5

    # Real ranked genes
    ax.plot(x, res["real"][valid], color=color, lw=2.5,
            marker="o", markersize=7.5, label=f"Ranked {disorder_name} genes", zorder=5)  # 5 * 1.5

    ax.axhline(0, color="gray", ls=":", lw=1, alpha=0.5)
    ax.set_xlabel("Number of added genes", fontsize=18)  # 12 * 1.5
    ax.set_title(disorder_name, fontweight="bold", fontsize=21)  # 14 * 1.5
    ax.legend(fontsize=12, framealpha=0.8, loc="lower right")  # 8 * 1.5
    ax.set_ylim(-0.6, 1.02)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

axes[0].set_ylabel("Spearmans' R with top-61 bias", fontsize=18)  # 12 * 1.5

# Set axes backgrounds to be fully transparent
for ax in axes:
    ax.set_facecolor('none')

fig.tight_layout()
fig.patch.set_alpha(0)  # Ensures figure bg is transparent in all backends

fig.savefig(_figdir + "FigSX_gene_expansion_slice.pdf",
            dpi=300, bbox_inches='tight', transparent=True, facecolor='none')
fig.savefig(_figdir + "FigSX_gene_expansion_slice.png",
            dpi=300, bbox_inches='tight', transparent=True, facecolor='none')
print(f"Saved: {_figdir}FigSX_gene_expansion_slice.pdf/png")
plt.show()

In [ ]:
# Save sweep cache
if not (_SWEEP_CACHE.exists() and not FORCE_RERUN):
    _sw_data = {
        'SCZ': (Corr_SCZ, Pval_SCZ, Corr_SCZ_unweighted),
        'HIQ': (Corr_HIQ, Pval_HIQ, Corr_HIQ_unweighted),
        'LIQ': (Corr_LIQ, Pval_LIQ, Corr_LIQ_unweighted),
        'DDD': (Corr_DDD, Pval_DDD, Corr_DDD_unweighted),
    }
    with open(_SWEEP_CACHE, "wb") as f:
        pickle.dump(_sw_data, f)
    print(f"Saved sweep cache: {_SWEEP_CACHE}")

In [ ]:
# Save expansion/slice cache
if not (_EXP_CACHE.exists() and not FORCE_RERUN):
    _exp_data = {"all_results": all_results, "slice_results": slice_results}
    with open(_EXP_CACHE, "wb") as f:
        pickle.dump(_exp_data, f)
    print(f"Saved expansion/slice cache: {_EXP_CACHE}")

# Save plot data for Figures_Supp inline plotting
import pickle
PLOT_DATA_DIR = str(PROJ_DIR / "results" / "figures" / "plot_data") + "/"
os.makedirs(PLOT_DATA_DIR, exist_ok=True)

gene_sweep_data = {
    'GeneIdx': GeneIdx.tolist(),
    'SCZ': {
        'Corr': Corr_SCZ,
        'Corr_unweighted': Corr_SCZ_unweighted,
        'GeneSig': (-np.log10(np.array(GeneWeights["Pval"].values[GeneIdx], dtype=np.float64))).tolist(),
        'ylabel_corr': "Bias Correlation with Main SCZ Set",
    },
    'ASD_woID': {
        'Corr': Corr_HIQ,
        'Corr_unweighted': Corr_HIQ_unweighted,
        'GeneSig': (-np.log10(np.array(Spark_Denovo_sub["pDenovoWEST_Meta"].values[GeneIdx], dtype=np.float64))).tolist(),
        'ylabel_corr': "Bias Correlation with Main ASD Set",
    },
    'ASD_wID': {
        'Corr': Corr_LIQ,
        'Corr_unweighted': Corr_LIQ_unweighted,
        'GeneSig': (-np.log10(np.array(Spark_Denovo_sub["pDenovoWEST_Meta"].values[GeneIdx], dtype=np.float64))).tolist(),
        'ylabel_corr': "Bias Correlation with Main ASD ID Set",
    },
    'DDD': {
        'Corr': Corr_DDD,
        'Corr_unweighted': Corr_DDD_unweighted,
        'GeneSig': (-np.log10(np.array(DDD_Genes["denovoWEST_p_full"].values[GeneIdx], dtype=np.float64))).tolist(),
        'ylabel_corr': "Bias Correlation with Main DDD Set",
    },
}

gene_expansion_data = {
    'n_add_values': n_add_values.tolist(),
    'total_genes': (61 + n_add_values).tolist(),
    'disorders': {},
}
for disorder_name, gw_df, color in disorder_configs:
    res = all_results[disorder_name]
    gene_expansion_data['disorders'][disorder_name] = {
        'real': res['real'].tolist(),
        'rand_mean': res['rand_mean'].tolist(),
        'rand_lo': res['rand_lo'].tolist(),
        'rand_hi': res['rand_hi'].tolist(),
        'color': res['color'],
    }

with open(PLOT_DATA_DIR + "gene_sweep_data.pkl", "wb") as f:
    pickle.dump(gene_sweep_data, f)
with open(PLOT_DATA_DIR + "gene_expansion_data.pkl", "wb") as f:
    pickle.dump(gene_expansion_data, f)
print(f"Saved plot data to {PLOT_DATA_DIR}")

# Print summary for all disorders
for disorder_name, _, _ in disorder_configs:
    res = all_results[disorder_name]
    print(f"\n=== {disorder_name}: Correlation with top-61 reference ===")
    print(f"{'N total':>8s}  {'Real':>7s}  {'Rand mean':>10s}  {'Rand 95%CI':>14s}")
    for i, n_add in enumerate(n_add_values):
        nt = 61 + n_add
        if n_add == 0:
            print(f"{nt:8d}  {res['real'][i]:7.3f}  {'—':>10s}  {'—':>14s}")
        else:
            print(f"{nt:8d}  {res['real'][i]:7.3f}  {res['rand_mean'][i]:10.3f}  "
                  f"[{res['rand_lo'][i]:.3f}, {res['rand_hi'][i]:.3f}]")

# DDD Verification: Real Top-200 vs Random-140 Permutation Test

The R3.2c figure shows DDD real top-200 genes don't clearly exceed random additions.
Here we run a direct permutation test:
1. Keep the top-61 DDD genes with their real BGMR-corrected weights
2. For the real set: add genes ranked 62-201 with their real weights → correlation with top-61 ref
3. For random draws: replace genes 62-201 with 140 random genes, using the same weight vector
4. Repeat 100 times → empirical p-value = fraction of random draws ≥ real correlation

In [ ]:
# --- DDD real top-201 correlation ---
# Reuse DDD_GW_full built in R3.2c above
n_ref = 61
n_add = 140
n_reps_perm = 100

# Reference: top-61 DDD bias
ref_gw_ddd = dict(zip(DDD_GW_full.index[:n_ref], DDD_GW_full["Weight"].values[:n_ref]))
ref_bias_ddd = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, ref_gw_ddd)
ref_bias_ddd = AnnotateCTDat(ref_bias_ddd, Anno)
ref_effect_ddd = ref_bias_ddd["EFFECT"].values
common_idx_ddd = ref_bias_ddd.index

# Real expansion: top-61 + ranked 62-201 (nonzero-weight only, same as R3.2c)
all_weights_ddd = DDD_GW_full["Weight"].values
added_pool_ddd = [(DDD_GW_full.index[i], all_weights_ddd[i])
                  for i in range(n_ref, len(DDD_GW_full)) if all_weights_ddd[i] > 0]
added_indices_ddd = [x[0] for x in added_pool_ddd]
added_weights_ddd = np.array([x[1] for x in added_pool_ddd])
n_real_added = min(n_add, len(added_pool_ddd))

print(f"DDD genes beyond top-61 with nonzero weight: {len(added_pool_ddd)}")
print(f"Using {n_real_added} real genes for expansion")

# Compute real top-(61+n_real_added) bias
real_gw = dict(zip(DDD_GW_full.index[:n_ref], all_weights_ddd[:n_ref]))
real_gw.update(dict(zip(added_indices_ddd[:n_real_added], added_weights_ddd[:n_real_added])))
real_bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, real_gw)
real_bias = AnnotateCTDat(real_bias, Anno)
real_r, _ = spearmanr(ref_effect_ddd, real_bias.loc[common_idx_ddd, "EFFECT"].values)
print(f"\nReal top-{61 + n_real_added} DDD correlation with top-61: rho = {real_r:.4f}")

# --- Random permutation draws ---
# Gene pool: all genes in expression matrix except top-200 of DDD
n_exclude = min(200, len(DDD_GW_full))
exclude_set_ddd = set(DDD_GW_full.index[:n_exclude].astype(int))
gene_pool_ddd = np.array([g for g in HumanCT_Z2_HCT.index.values if g not in exclude_set_ddd])

top61_idx_ddd = DDD_GW_full.index[:n_ref]
top61_wt_ddd = DDD_GW_full["Weight"].values[:n_ref]

def _one_ddd_random_trial(rep, n_use, seed):
    rng = np.random.default_rng(seed)
    rand_genes = rng.choice(gene_pool_ddd, size=n_use, replace=False)
    gw = dict(zip(top61_idx_ddd, top61_wt_ddd))
    gw.update(dict(zip(rand_genes, added_weights_ddd[:n_use])))
    bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, gw)
    bias = AnnotateCTDat(bias, Anno)
    r, _ = spearmanr(ref_effect_ddd, bias.loc[common_idx_ddd, "EFFECT"].values)
    return r

rand_rs = Parallel(n_jobs=10)(
    delayed(_one_ddd_random_trial)(rep, n_real_added, seed=42 + rep)
    for rep in range(n_reps_perm)
)
rand_rs = np.array(rand_rs)

# Empirical p-value: fraction of random ≥ real
p_empirical = np.mean(rand_rs >= real_r)
print(f"\n{'='*60}")
print(f"DDD Permutation Test: Top-{61+n_real_added} Real vs 61+{n_real_added} Random")
print(f"{'='*60}")
print(f"Real correlation:      rho = {real_r:.4f}")
print(f"Random mean:           rho = {rand_rs.mean():.4f} +/- {rand_rs.std():.4f}")
print(f"Random [2.5%, 97.5%]:  [{np.percentile(rand_rs, 2.5):.4f}, {np.percentile(rand_rs, 97.5):.4f}]")
print(f"Fraction random >= real: {p_empirical:.2f} ({int(np.sum(rand_rs >= real_r))}/{n_reps_perm})")
if p_empirical > 0.05:
    print("-> Real top-200 DDD genes do NOT significantly exceed random additions")
else:
    print("-> Real top-200 DDD genes significantly exceed random additions")

In [ ]:
# Histogram of random correlations vs real
fig, ax = plt.subplots(figsize=(7, 4), dpi=120)
ax.hist(rand_rs, bins=20, color="#2ca02c", alpha=0.6, edgecolor="white", label="Random draws (n=100)")
ax.axvline(real_r, color="red", lw=2.5, ls="--", label=f"Real top-{61+n_real_added} (rho={real_r:.3f})")
ax.axvline(rand_rs.mean(), color="gray", lw=1.5, ls=":", label=f"Random mean (rho={rand_rs.mean():.3f})")
ax.set_xlabel("Spearmans' R with top-61 DDD reference", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(f"DDD: Real top-{61+n_real_added} vs 61 + {n_real_added} random genes\n"
             f"p(random >= real) = {p_empirical:.2f}", fontsize=12, fontweight="bold")
ax.legend(fontsize=10, framealpha=0.8)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
fig.tight_layout()
fig.patch.set_alpha(0)
plt.show()

# DDD: Isolated Slice 62-200 Bias vs Top-61

Key question: does the ADDED slice (genes 62-200) itself carry concordant signal?
Compare bias from ONLY genes 62-200 vs bias from ONLY 140 random genes.

In [ ]:
# Load DDD gene weights (now p-value ordered)
DDD_GW_top285 = pd.read_csv(f"{GeneWeightDIR}/DDD.top285.gw.bgmr.csv",
                           index_col=0, header=None, names=["Weight"])

# Top-61 reference bias
DDD_MainBias = pd.read_csv(
    str(PROJ_DIR / "results/main_results/random/Centering/DDD_61_bias_addP.csv"),
    index_col=0)





In [ ]:
# --- Real slice 62-200: bias from ONLY those genes ---
real_slice_gw = DDD_GW_top285.iloc[60:200]["Weight"].to_dict()
real_slice_gw_nz = {k: v for k, v in real_slice_gw.items() if v > 0}
real_slice_bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, real_slice_gw_nz)
real_slice_bias = AnnotateCTDat(real_slice_bias, Anno)
real_slice_r, real_slice_p = GetSingeCellBiasCorr(real_slice_bias, DDD_MainBias)

print(f"Real genes 62-200 ONLY (n={len(real_slice_gw_nz)} nonzero-weight):")
print(f"  Bias correlation with top-61: rho = {real_slice_r:.4f}, p = {real_slice_p:.2e}")

# --- Random 140 genes: bias from ONLY random genes (no top-61 core) ---
n_rand = len(real_slice_gw_nz)
exclude_set_ddd2 = set(DDD_GW_top285.index[:200].astype(int))
gene_pool_ddd2 = np.array([g for g in HumanCT_Z2_HCT.index.values if g not in exclude_set_ddd2])
rank_weights = np.array(list(real_slice_gw_nz.values()))

def _rand_slice_trial(rep, n_use, rank_weights, gene_pool):
    rng = np.random.default_rng(42 + rep)
    rand_genes = rng.choice(gene_pool, size=n_use, replace=False)
    gw = dict(zip(rand_genes, rank_weights[:n_use]))
    bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, gw)
    bias = AnnotateCTDat(bias, Anno)
    r, _ = GetSingeCellBiasCorr(bias, DDD_MainBias)
    return r

rand_slice_rs = Parallel(n_jobs=10)(
    delayed(_rand_slice_trial)(rep, n_rand, rank_weights, gene_pool_ddd2)
    for rep in range(100)
)
rand_slice_rs = np.array(rand_slice_rs)

p_slice = np.mean(rand_slice_rs >= real_slice_r)
print(f"\nRandom {n_rand} genes ONLY (no top-61 core):")
print(f"  Mean rho = {rand_slice_rs.mean():.4f} +/- {rand_slice_rs.std():.4f}")
print(f"  95% CI: [{np.percentile(rand_slice_rs, 2.5):.4f}, {np.percentile(rand_slice_rs, 97.5):.4f}]")
print(f"\nFraction random >= real: {p_slice:.2f} ({int(np.sum(rand_slice_rs >= real_slice_r))}/100)")
if real_slice_r > np.percentile(rand_slice_rs, 97.5):
    print("-> Real slice 62-200 carries SIGNIFICANT concordant signal with top-61")
elif real_slice_r > rand_slice_rs.mean():
    print("-> Real slice 62-200 trends above random but NOT significant")
else:
    print("-> Real slice 62-200 does NOT exceed random genes")

In [ ]:
# Histogram: isolated slice comparison
fig, ax = plt.subplots(figsize=(7, 4), dpi=120)
ax.hist(rand_slice_rs, bins=20, color="#2ca02c", alpha=0.6, edgecolor="white",
        label=f"Random {n_rand} genes (n=100 draws)")
ax.axvline(real_slice_r, color="red", lw=2.5, ls="--",
           label=f"Real genes 62-200 (rho={real_slice_r:.3f})")
ax.axvline(rand_slice_rs.mean(), color="gray", lw=1.5, ls=":",
           label=f"Random mean (rho={rand_slice_rs.mean():.3f})")
ax.set_xlabel("Spearmans' R with top-61 DDD bias (slice ONLY, no core)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(f"DDD: Isolated slice 62-200 vs {n_rand} random genes\n"
             f"p(random >= real) = {p_slice:.2f}", fontsize=12, fontweight="bold")
ax.legend(fontsize=9, framealpha=0.8)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
fig.tight_layout()
fig.patch.set_alpha(0)
plt.show()